In [17]:
import requests
import pandas as pd
import time

In [18]:

def fetch_binance_ohlcv(symbol, interval="1d", limit=730):
    url = "https://api.binance.com/api/v3/klines"

    params = {
        "symbol": symbol,
        "interval": interval,
        "limit": limit
    }

    response = requests.get(url, params=params, timeout=10)
    response.raise_for_status()
    data = response.json()

    df = pd.DataFrame(data, columns=[
        "open_time",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "close_time",
        "quote_asset_volume",
        "number_of_trades",
        "taker_buy_base_asset_volume",
        "taker_buy_quote_asset_volume",
        "ignore"
    ])

    # Add symbol column
    df["symbol"] = symbol

    # Convert timestamps
    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms")
    df["close_time"] = pd.to_datetime(df["close_time"], unit="ms")

    # Convert numeric columns
    numeric_cols = [
        "open",
        "high",
        "low",
        "close",
        "volume",
        "quote_asset_volume",
        "number_of_trades",
        "taker_buy_base_asset_volume",
        "taker_buy_quote_asset_volume"
    ]

    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric)

    # Drop useless column
    df = df.drop(columns=["ignore"])

    return df


In [19]:
symbols = ["BTCUSDT", "ETHUSDT", "SOLUSDT"]

all_data = []

for symbol in symbols:
    df = fetch_binance_ohlcv(symbol, interval="1d", limit=730)
    all_data.append(df)
    time.sleep(0.2)

ohlcv_df = pd.concat(all_data, ignore_index=True)

print(ohlcv_df.head())
print(ohlcv_df["symbol"].value_counts())

   open_time      open      high       low     close       volume  \
0 2024-07-02  62900.00  63288.83  61806.28  62135.47  18573.11875   
1 2024-07-03  62135.46  62285.94  59400.00  60208.58  32160.11127   
2 2024-07-04  60208.57  60498.19  56771.00  57050.01  54568.77276   
3 2024-07-05  57050.02  57546.00  53485.93  56628.79  81348.24756   
4 2024-07-06  56628.79  58475.00  56018.00  58230.13  21651.31558   

               close_time  quote_asset_volume  number_of_trades  \
0 2024-07-02 23:59:59.999        1.160792e+09           1308816   
1 2024-07-03 23:59:59.999        1.945901e+09           1839372   
2 2024-07-04 23:59:59.999        3.168620e+09           2523236   
3 2024-07-05 23:59:59.999        4.515040e+09           3749255   
4 2024-07-06 23:59:59.999        1.235399e+09           1411065   

   taker_buy_base_asset_volume  taker_buy_quote_asset_volume   symbol  
0                   9453.17931                  5.907632e+08  BTCUSDT  
1                  15055.45492        